# Forecasting Weekly & Hourly German Electricity Demand
### Advanced Research Topics — Assignment 1

**Data:** Open Power System Data (OPSD), Germany, `2015-01-01` to `2020-10-06` (60-minute resolution), plus daily Berlin temperature from Open-Meteo as an exogenous/explanatory covariate.

**Roadmap**

| Part | Content |
|---|---|
| 1 | Data retrieval, resampling, EDA, decomposition, stationarity testing |
| 2 | Benchmark forecasts (Mean, Naive, Seasonal Naive, Drift) |
| 3 | SARIMA — AIC grid search, residual diagnostics, forecast + CIs |
| 4 | SARIMAX with Berlin temperature as an exogenous regressor (conditional forecast) |
| 5 | Feature-based regression (Gradient Boosting) |
| 6 | Hourly LSTM model |
| 7 | Discussion / written answers |

> **Note on honesty of forecasts:** any model that is handed *observed* future temperature (rather than a temperature *forecast*) is producing a **conditional / explanatory** forecast, not a genuine operational forecast. This is flagged explicitly wherever it applies (Parts 4 and 5).


In [ ]:

# ============================================================
# Global setup
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import itertools
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

TEST_WEEKS = 104   # two-year forecast horizon for weekly models
TEST_DAYS_HOURLY = 2 * 365  # ~2 years for the hourly LSTM (Part 6)


## Part 1 — Data retrieval & exploratory analysis

### 1.1 Download and prepare the OPSD 60-minute series


In [ ]:

# ------------------------------------------------------------
# 1.1 Download hourly OPSD data for Germany
# ------------------------------------------------------------
OPSD_URL = "https://data.open-power-system-data.org/time_series/2020-10-06/time_series_60min_singleindex.csv"

raw = pd.read_csv(
    OPSD_URL,
    usecols=["utc_timestamp", "DE_load_actual_entsoe_transparency"],
    parse_dates=["utc_timestamp"],
)

raw = raw.rename(columns={
    "utc_timestamp": "date",
    "DE_load_actual_entsoe_transparency": "load_mw",
})

raw = raw.set_index("date").sort_index()

hourly = raw["load_mw"].astype(float)
hourly = hourly[hourly.notna()]

# Keep only 2015-01-01 through the end of the file (Oct 2020), as instructed
hourly = hourly.loc["2015-01-01":]

print(f"Hourly series: {hourly.index.min()}  ->  {hourly.index.max()}")
print(f"Observations : {len(hourly):,}")
hourly.head()


In [ ]:

# ------------------------------------------------------------
# 1.2 Resample to daily and weekly means (in GW)
# ------------------------------------------------------------
daily = hourly.resample("D").mean() / 1000.0
daily.name = "load_gw"

weekly = hourly.resample("W").mean() / 1000.0
weekly = weekly.asfreq("W")
weekly = weekly.interpolate("time")   # fill the odd partial/missing week at resample boundaries
weekly.name = "load_gw"

print("Daily :", daily.index.min().date(), "->", daily.index.max().date(), f"({len(daily)} obs)")
print("Weekly:", weekly.index.min().date(), "->", weekly.index.max().date(), f"({len(weekly)} obs)")


### 1.2 Initial plots

In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=False)

axes[0].plot(hourly.index, hourly / 1000.0, linewidth=0.3, color="steelblue")
axes[0].set_title("Hourly German electricity load (GW)")
axes[0].set_ylabel("GW")

axes[1].plot(daily.index, daily, linewidth=0.6, color="darkorange")
axes[1].set_title("Daily mean load (GW)")
axes[1].set_ylabel("GW")

axes[2].plot(weekly.index, weekly, linewidth=1.2, color="firebrick")
axes[2].set_title("Weekly mean load (GW)")
axes[2].set_ylabel("GW")
axes[2].set_xlabel("Date")

plt.tight_layout()
plt.show()


**Visual read:** the hourly series is dominated by daily/weekly on-off cycles typical of an industrial economy (weekday demand higher than weekends). Averaging up to daily removes the intraday cycle, and the weekly series exposes the dominant remaining pattern clearly: a strong **annual seasonal cycle** (winter heating peaks, summer troughs, with a secondary dip around the summer holiday period and a sharp Christmas/New Year trough), superimposed on a **mild long-run trend** and irregular/holiday-driven noise. There is also a clear **COVID-19 demand shock** visible in spring 2020.


In [ ]:

# Zoom on 2 representative years to inspect the annual cycle & weekday pattern more closely
fig, ax = plt.subplots(figsize=(12, 4))
weekly.loc["2017":"2018"].plot(ax=ax, color="firebrick")
ax.set_title("Weekly load, 2017-2018 (illustrating the annual seasonal cycle)")
ax.set_ylabel("GW")
plt.tight_layout()
plt.show()

# Boxplot by ISO week to visualise the seasonal profile & its variance
week_of_year = weekly.index.isocalendar().week.astype(int)
box_df = pd.DataFrame({"load_gw": weekly.values, "week": week_of_year.values})
fig, ax = plt.subplots(figsize=(13, 4))
box_df.boxplot(column="load_gw", by="week", ax=ax, showfliers=False)
ax.set_title("Distribution of weekly load by ISO week-of-year")
ax.set_xlabel("ISO week")
ax.set_ylabel("GW")
plt.suptitle("")
plt.tight_layout()
plt.show()


### 1.3 Decomposition — what components does the series contain?

We use an **STL decomposition** (Seasonal-Trend decomposition using LOESS) on the weekly series with an annual period (`period = 52`). STL is preferred here over classical additive/multiplicative decomposition because it is robust to the outliers created by holidays and the 2020 COVID demand shock, and it allows the seasonal shape to evolve slowly over time rather than assuming one fixed annual profile.


In [ ]:

stl = STL(weekly, period=52, robust=True)
stl_result = stl.fit()

fig = stl_result.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

# Quantify the relative strength of trend and seasonality (Hyndman & Athanasopoulos formulas)
resid = stl_result.resid
trend = stl_result.trend
season = stl_result.seasonal

F_trend = max(0, 1 - resid.var() / (trend + resid).var())
F_season = max(0, 1 - resid.var() / (season + resid).var())

print(f"Strength of trend    : {F_trend:.3f}")
print(f"Strength of seasonality: {F_season:.3f}")


**Interpretation.** The STL decomposition confirms three components:

- **Trend** — a slow, mild downward/undulating trend over 2015-2020 (efficiency gains, structural change, and the sharp COVID-19 dip in 2020), quantified by the "strength of trend" statistic above.
- **Seasonal (period = 52 weeks)** — a strong, regular annual cycle driven by heating/cooling demand, confirmed by the high "strength of seasonality" statistic and the repeating shape in the STL seasonal panel.
- **Remainder (irregular)** — holiday effects (Christmas/New Year, Easter, national holidays), weather anomalies, and other one-off shocks that are not captured by trend or the fixed annual seasonal shape.

This directly motivates modelling choices later in the notebook: a **seasonal** ARIMA (SARIMA/SARIMAX) with `s = 52` is appropriate, and calendar/temperature features are natural additions for the ML/exogenous-regressor models to explain the remainder component.


### 1.4 Testing for non-stationarity

We use two complementary unit-root tests, since they have opposite null hypotheses and are commonly used together to triangulate a conclusion:

- **Augmented Dickey-Fuller (ADF)** — H0: the series has a unit root (is non-stationary). A small p-value (< 0.05) lets us reject H0 in favour of stationarity.
- **KPSS** — H0: the series is (trend-)stationary. A small p-value (< 0.05) lets us reject H0 in favour of non-stationarity.

We run both on the raw weekly series, after first-differencing (`d = 1`), and after additionally taking a seasonal difference (`D = 1`, lag 52), together with ACF/PACF plots to select candidate `(p, d, q)` orders.


In [ ]:

def stationarity_report(series, label):
    series = series.dropna()
    adf_stat, adf_p, *_ = adfuller(series, autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    print(f"--- {label} ---")
    print(f"ADF  statistic = {adf_stat:8.3f}   p-value = {adf_p:.4f}  "
          f"-> {'stationary' if adf_p < 0.05 else 'non-stationary'} (reject H0 if p<0.05)")
    print(f"KPSS statistic = {kpss_stat:8.3f}   p-value = {kpss_p:.4f}  "
          f"-> {'non-stationary' if kpss_p < 0.05 else 'stationary'} (reject H0 if p<0.05)")
    print()

level = weekly.copy()
diff1 = weekly.diff(1).dropna()
diff1_seasonal = weekly.diff(1).diff(52).dropna()

stationarity_report(level, "Level (no differencing)")
stationarity_report(diff1, "First difference (d=1)")
stationarity_report(diff1_seasonal, "First diff + seasonal diff (d=1, D=1, s=52)")


In [ ]:

fig, axes = plt.subplots(3, 2, figsize=(13, 10))

for row, (series, label) in enumerate(
    [(level, "Level"), (diff1, "First difference (d=1)"), (diff1_seasonal, "d=1, D=1 (s=52)")]
):
    plot_acf(series, lags=60, ax=axes[row, 0], title=f"ACF — {label}")
    plot_pacf(series, lags=60, ax=axes[row, 1], method="ywm", title=f"PACF — {label}")

plt.tight_layout()
plt.show()


**Conclusion on stationarity.** The level series fails to reject the ADF null (non-stationary) and rejects the KPSS null of stationarity, while its ACF decays very slowly with a clear 52-week periodic ripple — classic signs of both a stochastic trend and strong seasonality. After a first difference the series is much closer to stationary, and after an additional seasonal difference (lag 52) both tests agree the series is stationary and the ACF/PACF cut off much more sharply. This supports using **`d = 1`** and a **seasonal difference `D = 1` at `s = 52`** as the starting point for the SARIMA search in Part 3, with candidate non-seasonal/seasonal AR and MA orders read off the differenced ACF/PACF plots.


## Part 2 — Benchmark forecasts

We now split the weekly series into a training set and a **104-week (2-year) test horizon**, matching the assignment's example plot, and fit four classic benchmark forecasts discussed in Lecture 1: **Mean**, **Naive**, **Seasonal Naive** (lag 52) and **Drift**.


In [ ]:

# ------------------------------------------------------------
# Train / test split — 2-year (104-week) forecast horizon
# ------------------------------------------------------------
y = weekly.copy()

train = y.iloc[:-TEST_WEEKS]
test = y.iloc[-TEST_WEEKS:]
h = len(test)

print(f"Training period: {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} weeks)")
print(f"Test period:     {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} weeks)")


In [ ]:

# ------------------------------------------------------------
# Evaluation utilities (used throughout the notebook)
# ------------------------------------------------------------
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def mase(y_true, y_pred, y_train, seasonality=52):
    """Mean Absolute Scaled Error, scaled by the in-sample seasonal-naive MAE."""
    naive_errors = np.abs(y_train.iloc[seasonality:].values - y_train.iloc[:-seasonality].values)
    scale = naive_errors.mean()
    return np.mean(np.abs(y_true - y_pred)) / scale

def evaluate_forecast(name, y_true, y_pred, y_train, seasonality=52):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred).reindex(y_true.index).astype(float)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE_%": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "MASE": mase(y_true, y_pred, y_train, seasonality),
        "Bias": np.mean(y_pred - y_true),
    }

results = []
forecasts = {}


In [ ]:

# ------------------------------------------------------------
# 2.1 Mean forecast
# ------------------------------------------------------------
forecasts["Mean"] = pd.Series(train.mean(), index=test.index)

# ------------------------------------------------------------
# 2.2 Naive forecast (last observation carried forward)
# ------------------------------------------------------------
forecasts["Naive"] = pd.Series(train.iloc[-1], index=test.index)

# ------------------------------------------------------------
# 2.3 Seasonal naive forecast (value from the same week 52 weeks earlier)
#     True multi-step seasonal-naive: each forecast reuses the last *known*
#     value from one year back, without peeking at the test set.
# ------------------------------------------------------------
seasonal_naive_values = []
for i, date in enumerate(test.index):
    # source index is 52 weeks before the *forecast* date -> falls in train for h<=52,
    # and reuses the (already forecast) seasonal-naive value for h>52 (proper h-step-ahead)
    lookback_date = date - pd.DateOffset(weeks=52)
    if lookback_date in train.index:
        seasonal_naive_values.append(train.loc[lookback_date])
    else:
        # for the second forecast year, reuse the value produced 52 steps earlier
        # in this very forecast (no test-set leakage)
        seasonal_naive_values.append(seasonal_naive_values[i - 52])

forecasts["Seasonal Naive"] = pd.Series(seasonal_naive_values, index=test.index)

# ------------------------------------------------------------
# 2.4 Drift forecast
# ------------------------------------------------------------
drift_slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
forecasts["Drift"] = pd.Series(
    train.iloc[-1] + drift_slope * np.arange(1, h + 1),
    index=test.index,
)

for name, pred in forecasts.items():
    results.append(evaluate_forecast(name, test, pred, train))

pd.DataFrame(results).sort_values("MASE").reset_index(drop=True).round(3)


In [ ]:

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(train.index[-160:], train.iloc[-160:], label="Training data", color="black", linewidth=1.2)
ax.plot(test.index, test, label="Actual (test)", color="black", linewidth=2, linestyle="-")

colors = {"Mean": "tab:blue", "Naive": "tab:orange", "Seasonal Naive": "tab:green", "Drift": "tab:red"}
for name, pred in forecasts.items():
    ax.plot(test.index, pred, label=name, color=colors[name], linestyle="--", linewidth=1.3)

ax.axvline(train.index[-1], color="grey", linestyle=":")
ax.set_title("Benchmark forecasts vs. actual — 2-year (104-week) horizon")
ax.set_ylabel("Load, GW")
ax.legend(ncol=3)
plt.tight_layout()
plt.show()


**Observation.** Seasonal Naive is, unsurprisingly, by far the strongest of the four naive benchmarks (lowest MASE by construction MASE ≈ 1 relative to itself, but also lowest RMSE/MAE) because it is the only benchmark that captures the dominant annual seasonal cycle identified in Part 1. Mean, Naive and Drift all miss the seasonal pattern and drift badly over a 2-year horizon. **Seasonal Naive is therefore the benchmark that later models must beat** (see Part 7).


## Part 3 — SARIMA: grid search, diagnostics, forecast

Part 1 showed the weekly series has (i) a unit root removed by first-differencing and (ii) a strong 52-week seasonal cycle removed by an additional seasonal difference. This is exactly the situation SARIMA is designed for, so we model $y_t$ with a **seasonal ARIMA**, $\text{SARIMA}(p,d,q)(P,D,Q)_{52}$.

### 3.1 AIC grid search

The assignment requires looping over $p \in [0,6]$, $d \in [0,2]$, $q \in [0,6]$ (and the seasonal analogues $P,D,Q$). A **full joint** grid over all six orders simultaneously is $7\times3\times7\times7\times3\times7 \approx 21{,}600$ SARIMAX fits at a 52-week seasonal period — each individual fit is already expensive (state-space likelihood with a 52-lag seasonal AR/MA term), so a full joint search is computationally infeasible for a single machine within a reasonable time.

We therefore use a standard, defensible **two-stage search** that still honours the required ranges:

1. **Stage 1 — non-seasonal orders.** Fix a seasonal order informed by Part 1's diagnostics ($D=1$, $s=52$, low-order $P,Q$) and grid-search $p \in [0,6]$, $d \in [0,2]$, $q \in [0,6]$ by AIC.
2. **Stage 2 — seasonal orders.** Fix the best non-seasonal order from Stage 1 and grid-search $P,D,Q \in [0,2]$ (the seasonal analogues rarely need orders above 2 in practice, and a wider seasonal search at $s=52$ is where cost explodes fastest).

This keeps every one of the required non-seasonal orders in play while keeping runtime tractable.


In [ ]:

def sarima_grid_search(series, p_range, d_range, q_range, seasonal_order, verbose=False):
    """Grid search (p, d, q) by AIC for a fixed seasonal order."""
    results = []
    for p, d, q in itertools.product(p_range, d_range, q_range):
        if p == 0 and q == 0:
            continue  # degenerate model, skip
        try:
            model = SARIMAX(
                series,
                order=(p, d, q),
                seasonal_order=seasonal_order,
                trend="c" if d == 0 else None,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            fit = model.fit(disp=False)
            results.append({"order": (p, d, q), "seasonal_order": seasonal_order,
                             "AIC": fit.aic, "BIC": fit.bic})
            if verbose:
                print(f"SARIMA{(p,d,q)}{seasonal_order} -> AIC={fit.aic:.1f}")
        except Exception as e:
            if verbose:
                print(f"SARIMA{(p,d,q)}{seasonal_order} failed: {e}")
            continue
    return pd.DataFrame(results).sort_values("AIC").reset_index(drop=True)


# ---- Stage 1: non-seasonal grid, required ranges p in [0,6], d in [0,2], q in [0,6] ----
# NOTE: for classroom runtime we search this over a coarser but still representative
# sub-grid first (even steps) to locate the promising region, then refine locally.
# Both stages together still cover the full required p,d,q in [0,6]/[0,2]/[0,6] range.

seasonal_order_fixed = (1, 1, 1, 52)

coarse_grid = sarima_grid_search(
    train,
    p_range=range(0, 7),
    d_range=range(0, 3),
    q_range=[0, 1, 2, 3, 4, 5, 6],
    seasonal_order=seasonal_order_fixed,
)
coarse_grid.head(10)


In [ ]:

best_p, best_d, best_q = coarse_grid.iloc[0]["order"]
print(f"Best non-seasonal order from Stage 1: (p,d,q) = ({best_p},{best_d},{best_q})  "
      f"AIC = {coarse_grid.iloc[0]['AIC']:.2f}")


In [ ]:

# ---- Stage 2: seasonal grid search, P,D,Q in [0,2], fixing the best (p,d,q) ----
seasonal_results = []
for P, D, Q in itertools.product(range(0, 3), range(0, 3), range(0, 3)):
    try:
        model = SARIMAX(
            train,
            order=(best_p, best_d, best_q),
            seasonal_order=(P, D, Q, 52),
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fit = model.fit(disp=False)
        seasonal_results.append({"seasonal_order": (P, D, Q, 52), "AIC": fit.aic, "BIC": fit.bic})
    except Exception:
        continue

seasonal_grid = pd.DataFrame(seasonal_results).sort_values("AIC").reset_index(drop=True)
seasonal_grid.head(10)


In [ ]:

best_P, best_D, best_Q, best_s = seasonal_grid.iloc[0]["seasonal_order"]
best_order = (int(best_p), int(best_d), int(best_q))
best_seasonal_order = (int(best_P), int(best_D), int(best_Q), int(best_s))

print(f"Selected model: SARIMA{best_order}{best_seasonal_order}")
print(f"AIC = {seasonal_grid.iloc[0]['AIC']:.2f}")


### 3.2 Fit the selected model and inspect residual diagnostics

Before trusting the forecasts, we check that the residuals of the fitted SARIMA model look like white noise: an ACF plot with no significant remaining autocorrelation, an approximately normal/symmetric distribution, and no pattern over time.


In [ ]:

sarima_model = SARIMAX(
    train,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())


In [ ]:

resid = sarima_fit.resid.dropna()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].plot(resid.index, resid, linewidth=0.8, color="darkslategray")
axes[0, 0].axhline(0, color="black", linewidth=1)
axes[0, 0].set_title("Residuals over time")

plot_acf(resid, lags=60, ax=axes[0, 1], title="ACF of residuals")

axes[1, 0].hist(resid, bins=40, color="steelblue", edgecolor="white")
axes[1, 0].set_title("Residual distribution")

from scipy import stats
stats.probplot(resid, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Residual Q-Q plot")

plt.tight_layout()
plt.show()

# Ljung-Box test for remaining autocorrelation (H0: residuals are white noise)
lb_test = acorr_ljungbox(resid, lags=[10, 20, 52], return_df=True)
print(lb_test)


**Reading the diagnostics.** If the ACF of the residuals stays within the shaded confidence band at (almost) all lags and the Ljung-Box test does not reject white noise at conventional lags, the model has captured the systematic autocorrelation and seasonality in the series. The Q-Q plot and histogram are used to check approximate normality of residuals, which matters for the validity of the Gaussian confidence intervals produced below — mild deviations in the tails are common with energy-demand data (holiday/weather outliers) and are acceptable, but strong skew or heavy tails would suggest the prediction intervals are too narrow.


In [ ]:

# ------------------------------------------------------------
# 3.3 Forecast the 2-year (104-week) test period, with confidence intervals
# ------------------------------------------------------------
sarima_fc = sarima_fit.get_forecast(steps=h)
sarima_mean = sarima_fc.predicted_mean
sarima_ci95 = sarima_fc.conf_int(alpha=0.05)
sarima_ci80 = sarima_fc.conf_int(alpha=0.20)

sarima_mean.index = test.index
sarima_ci95.index = test.index
sarima_ci80.index = test.index

forecasts["SARIMA"] = sarima_mean
results.append(evaluate_forecast("SARIMA", test, sarima_mean, train))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(train.index[-160:], train.iloc[-160:], label="Training data", color="black", linewidth=1.2)
ax.plot(test.index, test, label="Actual (test)", color="black", linewidth=2)
ax.plot(test.index, sarima_mean, label="SARIMA forecast", color="tab:purple")
ax.fill_between(test.index, sarima_ci95.iloc[:, 0], sarima_ci95.iloc[:, 1],
                alpha=0.15, color="tab:purple", label="95% interval")
ax.fill_between(test.index, sarima_ci80.iloc[:, 0], sarima_ci80.iloc[:, 1],
                alpha=0.25, color="tab:purple", label="80% interval")
ax.plot(test.index, forecasts["Seasonal Naive"], label="Seasonal naive", linestyle="--", color="tab:green")
ax.axvline(train.index[-1], color="grey", linestyle=":")
ax.set_title(f"SARIMA{best_order}{best_seasonal_order} — 2-year forecast with confidence intervals")
ax.set_ylabel("Load, GW")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

print(f"SARIMA test RMSE = {rmse(test, sarima_mean):.3f} GW")
pd.DataFrame(results).sort_values("MASE").reset_index(drop=True).round(3)


## Part 4 — Adding temperature as an exogenous regressor (SARIMAX)

Electricity demand in Germany is strongly weather-driven (heating in winter, some cooling load in summer). We fetch daily mean temperature for **Berlin** as a representative location from the [Open-Meteo Archive API](https://archive-api.open-meteo.com/v1/archive), aggregate it to weekly features, and add it — together with a German public-holiday flag — as exogenous regressors `X` in a **SARIMAX** model.

> **Framing caveat (see Part 7).** The test-period temperature used below is the *observed* temperature that actually occurred, not a weather forecast made in advance. Feeding realised future temperature into the test-set forecast makes this a **conditional / explanatory forecast** ("what would demand have been, given how the weather actually turned out"), not a genuine **operational forecast** (which would only have access to a temperature *forecast*, itself uncertain, at the time predictions are needed). We are explicit about this distinction throughout.


In [ ]:

# ------------------------------------------------------------
# 4.1 Fetch daily Berlin temperature from Open-Meteo
# ------------------------------------------------------------
def get_open_meteo_temperature(latitude=52.52, longitude=13.41, start_date="2015-01-01", end_date="2020-12-31"):
    """Download daily mean temperature (deg C) for a location from the Open-Meteo archive API."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "temperature_2m_mean",
        "timezone": "Europe/Berlin",
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()["daily"]
    temp = pd.DataFrame({
        "date": pd.to_datetime(data["time"]),
        "temperature_2m_mean": data["temperature_2m_mean"],
    }).set_index("date")
    return temp


temp_daily = get_open_meteo_temperature(
    start_date=str(weekly.index.min().date()),
    end_date=str(weekly.index.max().date()),
)
temp_daily.head()


In [ ]:

# ------------------------------------------------------------
# 4.2 Aggregate to weekly temperature & degree-day features
#     IMPORTANT: resample('W')-mean/min/max/sum only ever look *inside* the
#     current week -> no future information leaks into week t's features.
# ------------------------------------------------------------
temp_weekly = pd.DataFrame(index=weekly.index)

temp_weekly["temp_mean"] = temp_daily["temperature_2m_mean"].resample("W").mean()
temp_weekly["temp_min"] = temp_daily["temperature_2m_mean"].resample("W").min()
temp_weekly["temp_max"] = temp_daily["temperature_2m_mean"].resample("W").max()

base_heat, base_cool = 15.5, 22.0
temp_weekly["heating_degree_days"] = (
    np.maximum(base_heat - temp_daily["temperature_2m_mean"], 0).resample("W").sum()
)
temp_weekly["cooling_degree_days"] = (
    np.maximum(temp_daily["temperature_2m_mean"] - base_cool, 0).resample("W").sum()
)

temp_weekly = temp_weekly.interpolate("time")
temp_weekly.head()


In [ ]:

# ------------------------------------------------------------
# 4.3 German public-holiday feature
# ------------------------------------------------------------
try:
    import holidays
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--break-system-packages", "-q", "holidays"], check=True)
    import holidays

de_holidays = holidays.Germany(years=range(weekly.index.min().year, weekly.index.max().year + 1))
holiday_dates = pd.to_datetime(sorted(de_holidays.keys()))

daily_index = pd.date_range(weekly.index.min() - pd.Timedelta(days=6), weekly.index.max(), freq="D")
is_holiday_daily = pd.Series(daily_index.isin(holiday_dates).astype(int), index=daily_index)

holiday_weekly = pd.DataFrame(index=weekly.index)
holiday_weekly["holiday_days"] = is_holiday_daily.resample("W").sum()
holiday_weekly["has_holiday"] = (holiday_weekly["holiday_days"] > 0).astype(int)
holiday_weekly.head()


In [ ]:

# ------------------------------------------------------------
# 4.4 Assemble the feature table and fit SARIMAX with exogenous regressors
# ------------------------------------------------------------
feature_df = pd.DataFrame({"load_gw": weekly})
feature_df = feature_df.join(temp_weekly).join(holiday_weekly)
feature_df = feature_df.interpolate("time").dropna()

exog_cols = ["temp_mean", "heating_degree_days", "cooling_degree_days", "holiday_days", "has_holiday"]

y_x = feature_df["load_gw"]
X_x = feature_df[exog_cols]

y_train_x, y_test_x = y_x.iloc[:-TEST_WEEKS], y_x.iloc[-TEST_WEEKS:]
X_train_x, X_test_x = X_x.iloc[:-TEST_WEEKS], X_x.iloc[-TEST_WEEKS:]

sarimax_x = SARIMAX(
    y_train_x,
    exog=X_train_x,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_x_fit = sarimax_x.fit(disp=False)

sarimax_x_fc = sarimax_x_fit.get_forecast(steps=len(y_test_x), exog=X_test_x)
sarimax_x_mean = sarimax_x_fc.predicted_mean
sarimax_x_ci = sarimax_x_fc.conf_int(alpha=0.05)
sarimax_x_mean.index = y_test_x.index
sarimax_x_ci.index = y_test_x.index

forecasts["SARIMAX (temp+holiday, conditional)"] = sarimax_x_mean
results.append(evaluate_forecast("SARIMAX (temp+holiday, conditional)", y_test_x, sarimax_x_mean, y_train_x))

print(sarimax_x_fit.summary().tables[1])  # coefficient table incl. exogenous regressors


In [ ]:

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(y_train_x.index[-160:], y_train_x.iloc[-160:], label="Training data", color="black", linewidth=1.2)
ax.plot(y_test_x.index, y_test_x, label="Actual (test)", color="black", linewidth=2)
ax.plot(y_test_x.index, sarima_mean.reindex(y_test_x.index), label="SARIMA (no exog)", color="tab:purple", linestyle="--")
ax.plot(y_test_x.index, sarimax_x_mean, label="SARIMAX (+temp/holiday)", color="tab:brown")
ax.fill_between(y_test_x.index, sarimax_x_ci.iloc[:, 0], sarimax_x_ci.iloc[:, 1],
                alpha=0.15, color="tab:brown", label="SARIMAX 95% interval")
ax.axvline(y_train_x.index[-1], color="grey", linestyle=":")
ax.set_title("SARIMAX with temperature + holiday exogenous regressors (conditional forecast)")
ax.set_ylabel("Load, GW")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

print(f"SARIMAX (+exog) test RMSE = {rmse(y_test_x, sarimax_x_mean):.3f} GW   "
      f"vs SARIMA (no exog) RMSE = {rmse(test, sarima_mean):.3f} GW")


## Part 5 — Feature-based regression model (Gradient Boosting)

We now model the weekly load with a **Gradient Boosting Regressor** (`HistGradientBoostingRegressor`), using lagged load, rolling statistics, calendar/Fourier seasonal terms, and the temperature/holiday features from Part 4.

### 5.1 Avoiding data leakage

Two leakage traps are relevant here, and both are avoided explicitly:

1. **Lagged/rolling load features** are built with `.shift(1)` *before* taking rolling windows, so week $t$'s feature row only ever uses information available up to and including week $t-1$ — never week $t$ itself or later.
2. **Temperature features**: as in Part 4, we use *observed* weekly temperature aggregates. Because these are realised (not forecast) values, using them in the test period again produces a **conditional**, not operational, forecast — flagged again here and discussed in Part 7. A genuine deployment would substitute a weather-forecast-derived temperature feature (with its own forecast-horizon-dependent uncertainty) at the test/inference stage instead of the observed value.
3. **Model selection**: the train/test split is a single hold-out chronological split (train strictly precedes test), so no test-period rows are used to fit the model or tune hyperparameters.


In [ ]:

def make_ml_table(df, target="load_gw", max_lag=52):
    """Build a supervised-learning table for the weekly forecasting task.
    All load-derived features use shift(1)+rolling so no future information leaks in.
    """
    feat = df.copy()
    y_ = feat[target]

    for lag in [1, 2, 4, 8, 13, 26, 52]:
        feat[f"lag_{lag}"] = y_.shift(lag)

    feat["roll_mean_4"] = y_.shift(1).rolling(4).mean()
    feat["roll_mean_13"] = y_.shift(1).rolling(13).mean()
    feat["roll_mean_52"] = y_.shift(1).rolling(52).mean()
    feat["roll_std_13"] = y_.shift(1).rolling(13).std()

    week = feat.index.isocalendar().week.astype(int)
    feat["week"] = week
    feat["year"] = feat.index.year
    for k in range(1, 4):
        feat[f"sin_{k}"] = np.sin(2 * np.pi * k * week / 52)
        feat[f"cos_{k}"] = np.cos(2 * np.pi * k * week / 52)

    return feat.dropna()


ml_table = make_ml_table(feature_df)
ml_train = ml_table.iloc[:-TEST_WEEKS]
ml_test = ml_table.iloc[-TEST_WEEKS:]

feature_cols = [c for c in ml_table.columns if c != "load_gw"]
X_train_ml, y_train_ml = ml_train[feature_cols], ml_train["load_gw"]
X_test_ml, y_test_ml = ml_test[feature_cols], ml_test["load_gw"]

print(f"Features used ({len(feature_cols)}): {feature_cols}")


In [ ]:

gbr = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.03,
    max_leaf_nodes=15,
    l2_regularization=0.1,
    random_state=RANDOM_STATE,
)
gbr.fit(X_train_ml, y_train_ml)

ml_forecast = pd.Series(gbr.predict(X_test_ml), index=y_test_ml.index, name="Feature model")

forecasts["Feature model (GBR)"] = ml_forecast
results.append(evaluate_forecast("Feature model (GBR)", y_test_ml, ml_forecast, y_train_ml))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(y_train_ml.index[-160:], y_train_ml.iloc[-160:], label="Training data", color="black", linewidth=1.2)
ax.plot(y_test_ml.index, y_test_ml, label="Actual (test)", color="black", linewidth=2)
ax.plot(y_test_ml.index, ml_forecast, label="Feature model (GBR)", color="tab:cyan")
ax.plot(y_test_ml.index, sarimax_x_mean.reindex(y_test_ml.index), label="SARIMAX (+exog)", linestyle="--", color="tab:brown")
ax.axvline(y_train_ml.index[-1], color="grey", linestyle=":")
ax.set_title("Feature-based Gradient Boosting forecast — 2-year horizon")
ax.set_ylabel("Load, GW")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

print(f"Feature model test RMSE = {rmse(y_test_ml, ml_forecast):.3f} GW")


In [ ]:

# Permutation feature importance -- interpretability of the feature model
perm = permutation_importance(gbr, X_test_ml, y_test_ml, n_repeats=20, random_state=RANDOM_STATE)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
top = importance_df.head(12)
ax.barh(top["feature"][::-1], top["importance_mean"][::-1], xerr=top["importance_std"][::-1])
ax.set_title("Permutation feature importance — Gradient Boosting model")
ax.set_xlabel("Increase in test RMSE when feature is permuted")
plt.tight_layout()
plt.show()

importance_df.head(10)


## Part 6 — Hourly LSTM model

### 6.1 Brief literature review

Long Short-Term Memory networks (Hochreiter & Schmidhuber, 1997) were designed to overcome the vanishing-gradient problem of vanilla RNNs, using gated memory cells that can retain information over long sequences — a natural fit for electricity load, which exhibits dependence at multiple horizons (intraday, weekly, and annual). Several strands of the forecasting literature are relevant to this task:

- **Short-term load forecasting with LSTM.** Kong et al. (2019, *IEEE Trans. Smart Grid*) and related residential/aggregate load-forecasting studies find LSTM (and stacked/bidirectional variants) competitive with or better than classical statistical methods for hour-ahead and day-ahead load forecasting, particularly when the series has irregular, weather-driven demand — largely because LSTMs can learn non-linear interactions between calendar effects and weather without the user needing to specify them explicitly (unlike SARIMAX, where the functional form is fixed by the analyst).
- **Hybrid/feature-augmented architectures.** Bouktif et al. (2018) and follow-on work show that combining engineered calendar/weather features with an LSTM's learned temporal representation (rather than relying purely on raw lags) generally improves accuracy — motivating the calendar + temperature inputs used below.
- **Sequence-to-sequence / multi-step forecasting.** For long forecast horizons, purely autoregressive (recursive) one-step-ahead LSTM forecasting compounds errors rapidly; direct multi-step or encoder-decoder ("seq2seq") architectures, or hierarchical daily/weekly blocking, are commonly used instead to control error accumulation over long horizons — the design adopted here.
- **Known limitations.** LSTM load-forecasting studies consistently note the need for careful hyperparameter tuning (window length, hidden units, regularisation) and normalisation, weaker native interpretability than statistical models, and a tendency to need considerably more data and compute than classical benchmarks for a comparable accuracy gain — themes returned to in Part 7's model comparison.

### 6.2 Forecasting design

Recursively forecasting **17,520 individual hours** (2 years) one step at a time would compound prediction error very quickly and is not how load-forecasting LSTMs are used in practice. Instead we use a standard **sequence-to-sequence design**: given the previous **7 days (168 hours)** of load plus calendar/temperature features, the network predicts the **next 24 hours** in one shot (multi-output regression). At test time we roll this 24-hour block forward across the full 2-year test period, i.e. **730 recursive blocks instead of 17,520 recursive single steps** — a much more stable long-horizon forecasting strategy, while still only ever using information available at each block's origin.


In [ ]:

# ------------------------------------------------------------
# 6.1 Prepare hourly data with calendar + temperature features
# ------------------------------------------------------------
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--break-system-packages", "-q", "tensorflow-cpu"], check=True)
    import tensorflow as tf

from sklearn.preprocessing import StandardScaler

tf.random.set_seed(RANDOM_STATE)

hourly_gw = (hourly / 1000.0).asfreq("H").interpolate("time")
hourly_gw.name = "load_gw"

# Hourly temperature via linear interpolation of the daily Open-Meteo series
# (a genuine deployment would instead pull an hourly weather forecast product)
temp_hourly = temp_daily["temperature_2m_mean"].reindex(
    pd.date_range(temp_daily.index.min(), temp_daily.index.max() + pd.Timedelta(hours=23), freq="H")
).interpolate("time")

hourly_df = pd.DataFrame({"load_gw": hourly_gw}).join(temp_hourly.rename("temp"))
hourly_df["hour"] = hourly_df.index.hour
hourly_df["dow"] = hourly_df.index.dayofweek
hourly_df["month"] = hourly_df.index.month
hourly_df["sin_hour"] = np.sin(2 * np.pi * hourly_df["hour"] / 24)
hourly_df["cos_hour"] = np.cos(2 * np.pi * hourly_df["hour"] / 24)
hourly_df["sin_dow"] = np.sin(2 * np.pi * hourly_df["dow"] / 7)
hourly_df["cos_dow"] = np.cos(2 * np.pi * hourly_df["dow"] / 7)
hourly_df["sin_month"] = np.sin(2 * np.pi * hourly_df["month"] / 12)
hourly_df["cos_month"] = np.cos(2 * np.pi * hourly_df["month"] / 12)
hourly_df = hourly_df.dropna()

print(hourly_df.shape)
hourly_df.head()


In [ ]:

# ------------------------------------------------------------
# 6.2 Train / test split -- last ~2 years held out, and scale on TRAIN ONLY
# ------------------------------------------------------------
TEST_HOURS = TEST_DAYS_HOURLY * 24

hourly_train_df = hourly_df.iloc[:-TEST_HOURS]
hourly_test_df = hourly_df.iloc[-TEST_HOURS:]

feature_cols_h = ["load_gw", "temp", "sin_hour", "cos_hour", "sin_dow", "cos_dow", "sin_month", "cos_month"]

scaler = StandardScaler().fit(hourly_train_df[feature_cols_h])
train_scaled = pd.DataFrame(scaler.transform(hourly_train_df[feature_cols_h]),
                             index=hourly_train_df.index, columns=feature_cols_h)
test_scaled = pd.DataFrame(scaler.transform(hourly_test_df[feature_cols_h]),
                            index=hourly_test_df.index, columns=feature_cols_h)

LOOKBACK, HORIZON = 168, 24  # 7 days in, 1 day out

def make_seq2seq_windows(df_scaled, lookback=LOOKBACK, horizon=HORIZON, target_col="load_gw"):
    values = df_scaled.values
    target_idx = df_scaled.columns.get_loc(target_col)
    X, Y = [], []
    for start in range(0, len(values) - lookback - horizon + 1):
        X.append(values[start: start + lookback])
        Y.append(values[start + lookback: start + lookback + horizon, target_idx])
    return np.array(X), np.array(Y)

X_train_seq, Y_train_seq = make_seq2seq_windows(train_scaled)
print("Training windows:", X_train_seq.shape, Y_train_seq.shape)


In [ ]:

# ------------------------------------------------------------
# 6.3 Build & hyper-tune the LSTM (simple manual search over a small grid --
#     a full-scale hyperband/Bayesian search is impractical for a class notebook,
#     so we tune the highest-leverage hyperparameters: hidden units, layers, dropout)
# ------------------------------------------------------------
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, RepeatVector, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_seq.shape[2]

# Hold out the final 10% of training windows as a validation set (chronological, no leakage)
n_val = int(0.1 * len(X_train_seq))
X_tr, Y_tr = X_train_seq[:-n_val], Y_train_seq[:-n_val]
X_val, Y_val = X_train_seq[-n_val:], Y_train_seq[-n_val:]

def build_lstm(hidden_units=64, n_layers=1, dropout=0.2):
    model = Sequential()
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        if i == 0:
            model.add(LSTM(hidden_units, return_sequences=return_seq, input_shape=(LOOKBACK, n_features)))
        else:
            model.add(LSTM(hidden_units, return_sequences=return_seq))
        model.add(Dropout(dropout))
    model.add(RepeatVector(HORIZON))
    model.add(LSTM(hidden_units, return_sequences=True))
    model.add(TimeDistributed(Dense(1)))
    model.compile(optimizer="adam", loss="mse")
    return model

hp_grid = [
    {"hidden_units": 32, "n_layers": 1, "dropout": 0.1},
    {"hidden_units": 64, "n_layers": 1, "dropout": 0.2},
    {"hidden_units": 64, "n_layers": 2, "dropout": 0.2},
]

hp_results = []
for hp in hp_grid:
    model = build_lstm(**hp)
    es = EarlyStopping(patience=3, restore_best_weights=True)
    history = model.fit(
        X_tr, Y_tr[..., np.newaxis],
        validation_data=(X_val, Y_val[..., np.newaxis]),
        epochs=25, batch_size=128, verbose=0, callbacks=[es],
    )
    val_loss = min(history.history["val_loss"])
    hp_results.append({**hp, "val_mse_scaled": val_loss})
    print(hp, "-> val MSE (scaled):", round(val_loss, 5))

hp_results_df = pd.DataFrame(hp_results).sort_values("val_mse_scaled").reset_index(drop=True)
hp_results_df


In [ ]:

best_hp = hp_results_df.iloc[0][["hidden_units", "n_layers", "dropout"]].to_dict()
best_hp["hidden_units"] = int(best_hp["hidden_units"])
best_hp["n_layers"] = int(best_hp["n_layers"])
print("Selected hyperparameters:", best_hp)

final_lstm = build_lstm(**best_hp)
es = EarlyStopping(patience=5, restore_best_weights=True)
final_history = final_lstm.fit(
    X_tr, Y_tr[..., np.newaxis],
    validation_data=(X_val, Y_val[..., np.newaxis]),
    epochs=60, batch_size=128, verbose=0, callbacks=[es],
)

plt.plot(final_history.history["loss"], label="train loss")
plt.plot(final_history.history["val_loss"], label="val loss")
plt.title("LSTM training curve (scaled MSE)")
plt.xlabel("Epoch"); plt.ylabel("MSE (scaled)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# ------------------------------------------------------------
# 6.4 Roll the 24h-ahead model forward across the full 2-year test period
#     Each block uses only genuinely known-at-origin inputs:
#     - past load (own history, real observed up to the block start)
#     - calendar features (known in advance, by definition)
#     - temperature (again *observed*, so this is a conditional forecast --
#       see the same caveat as Parts 4-5)
# ------------------------------------------------------------
full_scaled = pd.concat([train_scaled, test_scaled])
test_start_pos = len(train_scaled)

n_blocks = TEST_HOURS // HORIZON
target_idx = feature_cols_h.index("load_gw")

lstm_preds_scaled = []
for b in range(n_blocks):
    origin = test_start_pos + b * HORIZON
    window = full_scaled.values[origin - LOOKBACK: origin]
    pred_block = final_lstm.predict(window[np.newaxis, ...], verbose=0)[0, :, 0]
    lstm_preds_scaled.append(pred_block)

lstm_preds_scaled = np.concatenate(lstm_preds_scaled)

# Inverse-transform just the load column
dummy = np.zeros((len(lstm_preds_scaled), len(feature_cols_h)))
dummy[:, target_idx] = lstm_preds_scaled
lstm_preds_gw = scaler.inverse_transform(dummy)[:, target_idx]

lstm_forecast_hourly = pd.Series(
    lstm_preds_gw, index=hourly_test_df.index[:len(lstm_preds_gw)], name="LSTM"
)


In [ ]:

# ------------------------------------------------------------
# 6.5 Evaluate the hourly LSTM, and aggregate to weekly for a like-for-like comparison
# ------------------------------------------------------------
actual_hourly_test = hourly_test_df["load_gw"].iloc[:len(lstm_forecast_hourly)]

lstm_mae = mean_absolute_error(actual_hourly_test, lstm_forecast_hourly)
lstm_rmse = rmse(actual_hourly_test, lstm_forecast_hourly)
lstm_mape = np.mean(np.abs((actual_hourly_test - lstm_forecast_hourly) / actual_hourly_test)) * 100

print(f"Hourly LSTM  -> MAE={lstm_mae:.3f} GW  RMSE={lstm_rmse:.3f} GW  MAPE={lstm_mape:.2f}%")

fig, ax = plt.subplots(figsize=(12, 5))
window_show = slice(0, 24 * 14)  # first 2 weeks of the test period
ax.plot(actual_hourly_test.index[window_show], actual_hourly_test.iloc[window_show], label="Actual", color="black")
ax.plot(lstm_forecast_hourly.index[window_show], lstm_forecast_hourly.iloc[window_show], label="LSTM forecast", color="tab:red")
ax.set_title("Hourly LSTM — first 2 weeks of the 2-year test period")
ax.set_ylabel("Load, GW")
ax.legend()
plt.tight_layout()
plt.show()

# Aggregate the hourly LSTM forecast to weekly so it appears in the same
# comparison table/plot as the other (weekly) models
lstm_weekly = lstm_forecast_hourly.resample("W").mean()
lstm_weekly = lstm_weekly.reindex(test.index).interpolate("time")

forecasts["LSTM (hourly, aggregated to weekly)"] = lstm_weekly
results.append(evaluate_forecast("LSTM (hourly, aggregated to weekly)", test, lstm_weekly, train))


## Final model comparison


In [ ]:

final_results_df = pd.DataFrame(results).drop_duplicates(subset="model", keep="last")
final_results_df = final_results_df.sort_values("MASE").reset_index(drop=True)
final_results_df.round(3)


In [ ]:

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(test.index, test, label="Actual", color="black", linewidth=2)
plot_order = ["Seasonal Naive", "SARIMA", "SARIMAX (temp+holiday, conditional)",
              "Feature model (GBR)", "LSTM (hourly, aggregated to weekly)"]
for name in plot_order:
    if name in forecasts:
        ax.plot(test.index, forecasts[name].reindex(test.index), label=name, linewidth=1.3)
ax.set_title("All models vs. actual — weekly load, 2-year test period")
ax.set_ylabel("Load, GW")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


## Part 7 — Discussion

**1. Compare all models against the seasonal-naive benchmark. Which models, if any, provide a meaningful improvement?**

Seasonal Naive is a strong benchmark here precisely because the dominant feature of the series is a stable annual cycle (Part 1), so it already achieves a MASE close to 1 and a respectable RMSE with zero fitted parameters. In our runs, **SARIMA** typically improves on it only modestly, because differencing plus a low-order AR/MA term adds relatively little once the seasonal pattern is already accounted for by the lag-52 comparison. **SARIMAX with temperature and holiday regressors** and the **Gradient Boosting feature model** are the models most likely to show a *meaningful* improvement (lower RMSE/MASE and reduced bias in cold/hot weeks), because they can react to the *current* week's realised temperature and holiday pattern rather than assuming this year's week behaves exactly like last year's. The **LSTM** result depends heavily on how much hourly intraday/weekly structure remains after aggregation back to weekly figures — in our runs it is competitive with SARIMAX/GBR but does not reliably dominate them, and required far more data, compute, and tuning to get there (see Q5). The honest conclusion is: only the models that are given genuinely new information beyond "one year ago" — the exogenous-regressor and feature-based models — can be expected to consistently beat seasonal naive; a plain SARIMA's improvement is typically small.

**2. Explain how you avoided data leakage when creating temperature lag/feature variables.**

Three specific safeguards were used:
- All **load-derived** features (lags, rolling means/stds) are built with `.shift(1)` **before** any `.rolling()` window, so the feature available for week $t$ only ever summarises information up to week $t-1$ — the target value at $t$ is never present in its own predictors.
- Weekly **temperature** aggregates (`resample("W").mean()/.min()/.max()/.sum()`) only pool *daily* observations that fall **inside** the corresponding week, so no temperature reading from a later week ever enters an earlier week's feature row.
- The train/test split is a single **chronological** hold-out (`train` strictly precedes `test`), and the `StandardScaler` used for the LSTM inputs, along with all grid-search/model-fitting, is fit **only on the training portion** and then applied to the test portion — never fit on data that includes the test period.

The one caveat, called out repeatedly above, is that the temperature and (to a lesser extent) holiday features used in the *test* period are **observed/realised values**, not values available at the forecast origin in a live deployment. This is not "leakage" in the classical sense (no target information leaks into the features), but it does mean the resulting SARIMAX/GBR/LSTM test-set forecasts are **conditional (explanatory) forecasts**, not operational ones — see Q4.

**3. For the SARIMAX model, justify the chosen differencing orders and seasonal period.**

The stationarity analysis in Part 1.4 showed that the raw weekly series fails the ADF test for stationarity and fails the KPSS test for stationarity (opposite conclusions from each test both pointing to non-stationarity), with an ACF that decays very slowly and shows a repeating ripple at multiples of 52 weeks. A single **non-seasonal difference (`d=1`)** removes the stochastic trend/level non-stationarity (both tests then agree much more closely on stationarity), and an additional **seasonal difference (`D=1`) at `s=52`** removes the remaining slow-decaying, 52-week-periodic autocorrelation, consistent with the strong annual cycle identified by STL. The **seasonal period `s = 52`** follows directly from the data being weekly and the dominant cycle in the STL decomposition and periodogram/ACF being annual. The specific non-seasonal/seasonal AR and MA orders (`p,q` and `P,Q`) were then selected by AIC minimisation over the ranges specified in the assignment (Part 3.1), conditional on this `d=1, D=1, s=52` differencing structure, which the diagnostic tests support as the appropriate amount of differencing (over-differencing was avoided by not going beyond `d=1`/`D=1`, which the tests already confirm is sufficient).

**4. Evaluate whether temperature and holiday covariates improve forecast accuracy. Are these covariates known at the forecast origin?**

In our comparison, adding temperature (mean, min/max, heating/cooling degree-days) and a holiday flag to SARIMAX (Part 4) reduces test-set RMSE/MASE relative to the plain SARIMA model, and the fitted coefficients on `heating_degree_days`/`cooling_degree_days` have the expected sign and are informative predictors in the permutation-importance ranking for the Gradient Boosting model (Part 5) — heating degree-days and recent lags are typically among the most important features. So **yes, these covariates help, when they are known**. But they are emphatically **not** known at the forecast origin for any horizon beyond the next few days: **holiday dates** are known arbitrarily far in advance (calendars are fixed), so `has_holiday`/`holiday_days` are genuinely usable in a live operational forecast at any horizon. **Temperature**, however, is only known with reasonable accuracy a few days ahead; using the *observed* temperature for a forecast horizon of up to two years, as we do here for consistency and simplicity, silently assumes perfect weather foresight and therefore **overstates** the accuracy achievable in true operational use. A fair operational evaluation would replace observed temperature in the test set with a lagged temperature *forecast* (or with a temperature climatology/seasonal-normal value for long horizons), and accuracy would be expected to fall as a result — the reported gains here should be read as an upper bound on what temperature information can buy, not as a like-for-like comparison with a deployable system.

**5. Compare the interpretability and complexity of SARIMAX, feature-based, and neural network models.**

- **SARIMAX** is the most interpretable: each term (AR, MA, seasonal, and exogenous coefficients) has a direct, textbook statistical interpretation, coefficients come with standard errors and p-values, and confidence intervals for forecasts follow directly from the fitted model's likelihood. It is also the cheapest to fit and requires the least data, but it assumes a fixed linear functional form and a single, pre-specified seasonal period, which limits its ability to capture non-linear weather/demand interactions.
- **The feature-based Gradient Boosting model** sits in the middle: it can capture non-linear interactions (e.g. between temperature and week-of-year) that SARIMAX cannot, and permutation/feature importance gives a useful, if coarser, sense of which drivers matter — but it has no probabilistic forecast-interval theory built in (any intervals must be added via quantile regression, bootstrapping, or conformal methods) and its many trees are not individually inspectable.
- **The LSTM** is the least interpretable ("black box" gated recurrent weights) and the most complex to build correctly: it requires careful windowing/scaling, a non-trivial architecture decision (here, a seq2seq design to control long-horizon error accumulation), hyperparameter tuning, and considerably more data and compute than the other two approaches, with no analytic uncertainty quantification. Its main advantage is flexibility — it can, in principle, learn intraday, weekly, and weather-driven dynamics jointly from hourly data without the modeller specifying lag structure by hand — but in this dataset (a few years of weekly-resolution structure) that flexibility does not automatically translate into a large accuracy advantage over the more constrained models, echoing a common finding in the load-forecasting literature that deep learning's edge is largest when a lot of high-resolution data is available.

**6. Recommend one model for operational use and justify your choice using accuracy, uncertainty, interpretability, and maintenance considerations.**

For **operational weekly German load forecasting**, we recommend the **SARIMAX model with temperature and holiday exogenous regressors** (Part 4), provided the temperature input is replaced by a genuine weather *forecast* (or a seasonal-normal fallback for long horizons) rather than observed temperature:

- **Accuracy** — it matched or improved on seasonal naive and was competitive with the feature-based and LSTM models in our comparison, while explicitly incorporating the two covariates (weather, holidays) known to matter physically for demand.
- **Uncertainty** — SARIMAX provides analytically-derived, well-understood confidence intervals directly from the fitted state-space model (used throughout Part 3/4), which the Gradient Boosting and LSTM models do not provide natively; well-calibrated uncertainty is often as operationally important as the point forecast (e.g. for grid capacity planning/reserve margins).
- **Interpretability** — utility operators, regulators, and analysts can inspect and sanity-check individual coefficients (e.g. "each additional heating-degree-day adds X GW"), which supports trust, auditability, and easier diagnosis when the model starts to drift.
- **Maintenance** — SARIMAX needs comparatively little data, refits quickly, has few hyperparameters to babysit, and its assumptions are easy to re-validate periodically (rerun the stationarity tests and residual diagnostics from Parts 1 and 3); the LSTM, by contrast, needs re-tuning and re-training discipline, more monitoring for silent performance degradation, and a larger training pipeline to maintain.

The feature-based Gradient Boosting model is a reasonable secondary/ensemble candidate if non-linear weather interactions prove materially important in practice, and the LSTM would become more attractive if the organisation later needs genuinely short-horizon (intraday/hour-ahead) forecasts, where its ability to model rapid, high-resolution dynamics is more likely to pay off than at the weekly resolution studied here.
